# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Javiera Yáñez Sanchez

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [ ]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.14.2 environment at: /Users/javierayanez/Documents/Semestre 11/MDS7202-Laboratorio_Programacion_Cientifica/MDS7202/.venv
Resolved 140 packages in 1.37s                                       
⠙ Preparing packages... (0/92)                                                  
⠙ Preparing packages... (0/92)------------------     0 B/69.70 KiB           
⠙ Preparing packages... (0/92)------------------ 14.84 KiB/69.70 KiB         
⠙ Preparing packages... (0/92)------------------ 14.84 KiB/69.70 KiB         
pathspec             ------------------------------     0 B/55.98 KiB
⠙ Preparing packages... (0/92)------------------ 14.84 KiB/69.70 KiB         
pathspec             ------------------------------ 14.81 KiB/55.98 KiB
⠙ Preparing packages... (0/92)------------------ 14.84 KiB/69.70 KiB         
pathspec             ------------------------------ 14.81 KiB/55.98 KiB
⠙ Preparing packages... (0/92)------------------ 14.84 KiB/69.70 KiB         
pathspec             ----------

In [1]:
import time
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [2]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [3]:
# Escribe aquí tu código

# load_all_serial sobre todos los batches
df = load_all_serial(DATA_DIR, n_batches=20)

# Dtypes
print("Dtypes originales:")
print(df.dtypes)

# Memory usage: calcula cuantos bytes ocupa cada columna, luego los suma y transforma a Mib (redondea a 2 decimales)
print(f"\nMemoria original: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MiB")

# Conversion en copia de dataframe
df_opt = df.copy()  # .astype(...)

# float64 -> float32: columnas de audio features
float32_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "avg_artist_popularity",
]
df_opt[float32_cols] = df_opt[float32_cols].astype("float32")

# int64 -> int16
df_opt[["key", "mode"]] = df_opt[["key", "mode"]].astype("int16")

# int64 -> int32
df_opt[["year", "popularity", "duration_ms", "total_artist_followers"]] = df_opt[
    ["year", "popularity", "duration_ms", "total_artist_followers"]
].astype("int32")

# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

Dtypes originales:
id                         object
name                       object
album_name                 object
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                     object
year                        int64
genre                      object
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object

Memoria original: 414.25 MiB


### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

> 1. El formato Parquet esta orientado a almacenamiento de columnas de manera comprimida y optimizada. A diferencia de CSV (formato plano) Parquet es binario, por lo que se guarda en bytes optimizados para que las maquinas lo entiendan (no legible para humanos), ofreciendo eficiencia de espacio; preserva el tipo de dato exacto; permite guardar datos anidados o repetidos (no es soportado por csv); y reduce costos en la nube por su formato que es comprimido. En columnar storage, los valores de cada columna se agrupan en bloques contiguos en disco; consultas que leen pocas columnas solo acceden a esos bloques, reduciendo I/O.

> 2. Apache Arrow es un formato de memoria columnar estandar en memoria ram para procesamiento de alta velocidad. Se relaciona con Parquet porque Arrow es el formato en memoria y Parquet es en disco; pd.read_parquet usa Arrow internamente para deserializar sin copias extras; con CSV pandas debe parsear texto fila a fila; con Parquet la lectura es directa y mucho mas rapida.

> 3. float32 existe porque en ML la precision de 7 digitos es mas que suficiente para features de audio entre 0 y 1, ahorra memoria y es el tipo nativo de GPUs, acelerando el entrenamiento. Esa perdida de precision es irrelevante en metricas normalizadas, o diferencias que no se vean afectadas mas alla del sexto decimal.

> 4. No conviene reducir precision de tipo numerico cuando los valores tienen rango muy amplio (overflow en float32 para valores >3.4e8), en calculos de alta precision, o cuando los errores de redondeo se acumulan en operaciones iterativas.

> 5. Dos alternativas mas eficientes que pandas son Polars (usa Apache Arrow nativo) y DuckDB (base de datos analitica en proceso, consultas SQL directamente sobre Parquet). Los riesgos que tienen es que se deben aprender nuevamente las expresiones por el cambio de sintaxis (realentizacion), la compatibilidad en ML por lo general requiere pandas por lo que habria que cambiar a panda, y al ser tecnologias mas modernas puede ser dificil encontrar solucion a errores complejos. 

> 6. La reducción fue de 414.2 MiB a 401.3 MiB (3.1%). Es una reduccion leve porque la mayor parte de la memoria la consumen columnas de texto (lyrics, name, artists) que no se pueden comprimir con cambio de tipo numerico.

> 7. Reducir valence a float16 introduciria pérdida de precision significativa (solo 3 dígitos decimales). Los errores de redondeo afectarian la calidad del target del modelo, potencialmente degradando el RMSE de prediccion.

In [4]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [5]:
# Escribe aquí tu código

from concurrent.futures import ThreadPoolExecutor


def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    # lee todos los archivos Parquet de data_dir en paralelo
    paths = sorted(data_dir.glob("*.parquet"))  # busca todos los archivos parquet
    if n_batches is not None:
        paths = paths[:n_batches]
    with ThreadPoolExecutor(
        max_workers=None
    ) as executor:  # max workers = none decide python cuantos hilos usar (with cierra hilos)
        dfs = list(
            executor.map(load_batch, [str(p) for p in paths])
        )  # ejecucion de load_batch en paralelo sobre cada archivo (entrega lista de df)
    return pd.concat(dfs, ignore_index=True)  # concatena df

**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [6]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

> 1. I/O-bound significa que la operación está limitada por la velocidad de entrada/salida (disco, red). En cambio, CPU-bound está limitado por el procesador. La lectura de archivos es I/O-bound, el cuello de botella es la velocidad del disco, no la CPU.

> 2. El GIL es un mecanismo de exclusion mutua que permite que solo un hilo ejecute bytecode Python a la vez. Existe para simplificar la mplementacion del interprete de CPython y para garantizar la seguridad de hilos. Resuelve race conditions en gestion de memoria (utiliza un sistema de conteo de referencias para liberar memoria), pero limita el paralelismo real en tareas CPU-bound.

> 3. Usamos Python como lenguaje de pegamento. Las operaciones costosas (numpy, arrow, pytorch) se implementan en c/c++ y liberan el GIL durante su ejecucion, liberar GIL se logra paralelismo real con codigo Python legible.

> 4. ThreadPoolExecutor es para tareas I/O-bound porque cuando un hilo se bloquea, GIL se libera automaticamente y permite que otro hilo tome el control provechando el tiempo muerto. ProcessPoolExecutor es para tareas CPU-bound porque usa procesos separados sin GIL compartido (independientes), logrando paralelismo real.

> 5. Crear un pool de threads tiene overhead de creacion y sincronizacion. Con archivos de 1 kb, ese overhead seria mayor que el tiempo de lectura, haciendo la version paralela mas lenta que la serial.

> 6. En este entorno, si se observa una mejora con la lectura paralela; se comienza a notar desde los 6 archivos, donde se ve un salto (hacia arriba) para la lectura serial mientras que en paralelo se mantiene.

> 7. El speedup no es igual al numero de threads porque: overhead de crear y sincronizar threads consume tiempo de CPU; hay un cuello de botella en I/O (un solo disco y tiene un limite de velocidad de lectura maxima); pd.concat es secuencial; solo 1 CPU disponible en este entorno.

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [7]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [9]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [10]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

In [11]:
pd.__version__

'2.3.3'

> 1. La vectorizacion en numpy aplica operaciones sobre arrays enteros sin loops de Python. Puede ejecutar operaciones sin loops explicitos ya que  ejecuta en codigo C compilado con SIMD, procesando multiples elementos por ciclo de reloj.

> 2. JIT compila codigo en tiempo de ejecucion. @numba.njit compila la funcion a código maquina puro. El modo nopython=True exige compilación completa sin recurrir al interprete python, y si encuentra aglo que no se puede traducir tira error.

> 3. Numba es mas lento en la primera ejecución porque necesita compilar la funcion (warm-up de JIT). En el benchmark, se realiza una llamada inicial con datos pequeños antes del loop de medición para eliminar este costo del timing.

> 4. Polars es una libreria de dataframes escrita en Rust usando Apache Arrow. Diseñada para cuando los datos necesitan mas memoria RAM o tardarían mucho tiempo procesandose con pandas. Polars lleva a cabo un procesamiento eficiente y en paralelo de datos tabulares grandes. Este a ganado popularidad por ser mucho más rapida que pandas en operaciones de groupby/aggregation (incluso mejor que versiones recientes de pandas).

> 5. Polars usa Rust (vs C en pandas); modelo de ejecucion es lazy, es decir, optimiza el plan antes de ejecutar, en cambio pandas utiliza Eager (paso a paso); maneja la memoria con Apache Arrow compartiendo la memoria sin copiar (zero-copy), a diferencia de pandas que copia datos frecuentemente en la memoria;  y multi-hilo nativo (paralelismo), en lugar de estar limitado por GIL como pandas.

> 6. Pandas tiene overhead de indexacion, verificacion de tipos y metadatos de dataframes que no existen en NumPy puro.

> 7. SIMD son instrucciones CPU que aplican una operacion a multiples datos simultaneamente (ej. sumar 8 floats en un puro ciclo). Numpy y Polars las aprovechan automaticamente via compiladores optimizados.

> 8. Conviene Numba sobre Numpy cuando hay loops con logica condicional compleja o accesos no contiguos. Polars sobre pandas para datasets grandes con operaciones de aggregation/groupby en produccion.

> 9. La implementacion mas rapida fue Numba-JIT para  todos los datos, con speedups por encima de 300x sobre Python puro. Era esperable porque compila el loop a codigo maquina con SIMD sin arrays intermedios.

> 10. Pandas fue mas lento que NumPy hasta las 21k filas aprox. porque el overhead de indices, alineacion y metadatos de DataFrame supera el beneficio del dot product vectorizado internamente, despues de esto Numpy es mas lento.

> 11. La ventaja de Numpy sobre Python puro empieza a ser evidente desde las primeras mediciones, se puede apreciar que Numpy es mas rapido que Python en todo momento (cant. de filas). Numba muestra ventaja desde las primeras mediciones post-warm-up, antes de esto resulta ser mas lento (antes de las 10 filas aprox).

> 12. Polars no fue mas eficiente que pandas para este caso (tienen tiempos muy similares en especial despues de las 22k filas). Polars es mejor en en groupby/filtrado, no en regresion lineal vectorial. Y la versión de pandas si influye, pandas 2.0+ (en este caso '2.3.3') tiene optimizaciones de memoria, aunque polars sigue ganando por su arquitectura en Rust. 

> 13. Numba puede superar a Numpy en bucles simples porque es más eficiente con la memoria (numpy crea copias), y Numba compila el codigo completo en un solo bloque, haciendo todo de un viaje sin usar excesivamente la memoria RAM.

> 14. Si incluyeramos la conversion, Numba y Polars serian más lentos para datos pequeños porque se ocupa mas tiempo transformando los datos que haciendo los cálulos. No existe este costo cuando en producción los datos ya vienen en formato NumPy/Arrow (de máquina) desde el pipeline de entrada, por lo que la conversion no existiria.

> 15. Para 100M filas elegiria Polars debido a su lazy execution, que optimiza memoria RAM y permite paralelizacion. Con GPU, usaria TensorFlow/PyTorch que paralelice masivamente la multiplicación matricial en miles de nucleos.


### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [13]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

n_jobs=1  → tiempo: 267.6s | RMSE: 0.1652
n_jobs=-1 → tiempo: 56.8s | RMSE: 0.1652


In [14]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

> 1. n_jobs controla el numero de trabajos paralelos; en random forest paraleliza la construcción de arboles. Utilizar n_jobs=-1 hace que se usen todos los cores (o núcleos) disponibles; y es secuencial.

> 2. Random Forest usa joblib con backend `loky` (procesos independientes), evitando el GIL, es decir, cada proceso tiene su propio interprete Python. Además, las extensiones C de NumPy liberan el GIL durante operaciones numericas.

> 3. El n_jobs=-1 mejoró el tiempo en casi 200 segundos, mientras que n_jobs=1 se tardó 267.6 segundos, n_jobs=-1 se tardó solo 56.8segundos. Con multiples nucleos reales, la mejora es proporcional al número de arboles paralelizables.

> 4. No fue proporcional, por que tengo 8 CPUs y el speedup deberia ser de 8x aprox; el speedup real fue de 267.6 / 56.8 ≈ 4.7x, es decir, poco más de la mitad del ideal. Esto se debe a que no todos los arboles se pueden construir completamente en paralelo , el overhead de comunicación entre procesos de joblib, y que algunos nucleos pueden estar ocupados con otros procesos del sistema operativo.

> 5. No hubo diferencia en RMSE (0.1652 para ambos). Era esperable, ya que n_jobs solo controla la paralelizacion, no el algoritmo. Con random_state=42 fijo, ambos modelos construyen exactamente los mismos arboles.

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


In [ ]:
# Escribe aquí tu código (copia el template y completa los TODOs)


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>